# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
Unit of analysis + time window

Unit of analysis: One row represents one content item for one client (client_hash_id and content_hash_id).

Feature window: February 2026 (2026-02-01 to 2026-02-28).

Label window: March 2026 (2026-03-01 to 2026-03-31).

The feature and label windows do not overlap, which helps prevent data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
Fields: feature / label / context / excluded

Features:
- February 2026 Google Search performance measures such as impressions, clicks, and other performance signals available before the label window.

Label:
- March 2026 outcome for the content item, such as whether it went dark (received zero clicks during the measured March period).

Context:
- client_hash_id
- content_hash_id

Excluded:
- Future March performance measures used to define the label.
- Any field that is directly derived from the March outcome, because it would cause label leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
import os
import duckdb
import getpass
from google.colab import userdata

def get_hf_token():
    # 1. Try Colab Secrets
    try:
        return userdata.get('HF_TOKEN')
    except Exception:
        pass

    # 2. Try Environment Variable
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    # 3. Fallback to manual input
    return getpass.getpass("Hugging Face token not found in Secrets. Please paste your READ token (hf_...): ")

con = duckdb.connect()
# Load httpfs for remote file access
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET enable_progress_bar = false")

token = get_hf_token()
if token:
    # Configure the secret using the HUGGINGFACE type
    con.execute(f"""
        CREATE OR REPLACE SECRET hf_auth (
            TYPE HUGGINGFACE,
            TOKEN '{token}'
        );
    """)
    print("Hugging Face secret configured.")
else:
    print("Warning: No token provided.")

# Using HTTPS URLs which are better supported for authenticated requests in DuckDB 1.3
BASE_URL = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main"
FACT = f"{BASE_URL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{BASE_URL}/dim_content.parquet')"
CLI = f"read_parquet('{BASE_URL}/dim_clients.parquet')"

print("Database setup complete.")

Hugging Face token not found in Secrets. Please paste your READ token (hf_...): ··········
Hugging Face secret configured.
Database setup complete.


In [24]:
query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) AS daily_rows
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
ORDER BY daily_rows DESC
LIMIT 10
"""

con.sql(query).show()

┌─────────────────────────┬──────────────────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │ daily_rows │
│         varchar         │         varchar          │   int64    │
├─────────────────────────┼──────────────────────────┼────────────┤
│ client_e547b89c05043229 │ content_f338440914b1ab00 │         28 │
│ client_e547b89c05043229 │ content_d0fa1bbfbc10caf8 │         28 │
│ client_e547b89c05043229 │ content_7e131483384291cf │         28 │
│ client_e547b89c05043229 │ content_4c1e972bec56132e │         28 │
│ client_e547b89c05043229 │ content_4e48bd81bb37eb4f │         28 │
│ client_e547b89c05043229 │ content_d801c9cbea694166 │         28 │
│ client_e547b89c05043229 │ content_2b0b854b03f0b101 │         28 │
│ client_3ffa76342f366962 │ content_32bdebcb01540202 │         28 │
│ client_e547b89c05043229 │ content_64cad58fc02e7605 │         28 │
│ client_e547b89c05043229 │ content_8307db52c961228c │         28 │
├─────────────────────────┴─────────────────────

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
import duckdb

In [8]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.